<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Machine Learning Fundamentals** · Week 2 · Trees, Ensembles and More · Lesson 12 of 24

# Week 2 Project: Compare Models on Medical Data

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals).

**Learning objective:** Compare seven classifiers fairly on a real medical dataset, choose one with good reasons, and explain the result responsibly.

## The data: breast-cancer screening

scikit-learn includes a real dataset from the University of Wisconsin: measurements of cell nuclei from **569** breast-tissue samples, each labelled **malignant** (cancer) or **benign** (not cancer). Doctors use tools like this to support — never replace — their judgement.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print(X.shape)
print(data.target_names)          # 0 = malignant, 1 = benign
print(y.value_counts())
X.iloc[:, :6].head()

Watch out: here **0 means malignant** and **1 means benign**. Always check what each label means before you interpret results.

## Step 1 — Explore

- How many malignant and benign samples are there? (212 and 357.)
- Compare the average `mean radius` of malignant and benign samples (`groupby(y)`).
- Draw a histogram of `mean radius` coloured by class.

## Step 2 — Split once, for everyone

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

## Step 3 — Compare seven models

`make_pipeline` glues a scaler and a model together so the scaler is always fitted on training data only (you will learn pipelines fully in Week 4).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

models = {
    "Baseline": DummyClassifier(),
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "k-NN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Decision tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Gradient boosting": GradientBoostingClassifier(random_state=42),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC()),
    "Naive Bayes": GaussianNB(),
}
scores = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    scores[name] = round(m.score(X_test, y_test), 3)
print(pd.Series(scores).sort_values(ascending=False))

You should find logistic regression on top (about **0.99**), SVM and k-NN close behind (about 0.98), the ensembles about 0.96, and the baseline at about **0.63**.

## Step 4 — Think before you choose

The test set has only 143 samples, so one more or one fewer correct answer moves accuracy by 0.7 points. Differences of one or two points may be **luck**. In Week 3 you will use cross-validation to compare models more reliably.

Also ask: **which mistake is worse?** Calling a cancer "benign" (a missed cancer) is far more dangerous than calling a benign sample "malignant" (which leads to more tests). Count the missed cancers for your top model:

In [ ]:
best = models["Logistic regression"]
pred = best.predict(X_test)
missed = ((y_test == 0) & (pred == 1)).sum()
false_alarms = ((y_test == 1) & (pred == 0)).sum()
print("Missed cancers:", missed, " False alarms:", false_alarms)

## Step 5 — Report (one page)

1. The task, the data (569 samples, 30 measurements) and what 0 and 1 mean.
2. A sorted table of the eight test accuracies.
3. Your chosen model and **three reasons** (accuracy, simplicity, explainability, speed).
4. The number of missed cancers and false alarms, and why that matters more than accuracy.
5. Limits: one hospital's data from the 1990s; a model must be validated locally before any clinical use; the decision always stays with a doctor.

## Marking guide (self-check)

| Area | Excellent work |
|---|---|
| Fairness | Same split and test set for every model |
| Preparation | Scaling only inside pipelines for scale-sensitive models |
| Judgement | Notes that small differences may be luck |
| Responsibility | Discusses missed cancers and human oversight |
| Communication | Clear table and plain-English conclusion |

**Takeaway:** No algorithm wins everywhere; compare several on the same split, consider which mistakes matter, and choose with reasons — not only with the highest number.

## Try it yourself

Complete Steps 1–5. Then change `random_state=42` in the split to 0, 1 and 2. Does the ranking of the models change? What does that teach you about trusting a single test score?

---
Finished? Go back to [Machine Learning Fundamentals on WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*